In [1]:
import pandas as pd
df = pd.read_csv("bat_posts_results.csv")
print(df.columns.tolist())
print(df.head(3).to_string())
print(df.dtypes)

['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning']
  row_type post_id  comment_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  text  triage  na_subtype  triage_reason  EX EMO COG  MD  bat_score                                                                                       EX_reasoning   

In [2]:
"""
Quick count script for bat_posts_results.csv

Reports:
  - row_type breakdown (post vs comment, if mixed)
  - per-construct YES/NO/other counts and percentages (EX, EMO, COG, MD)
  - bat_score distribution (0-4) and mean
  - cross-tab: how many posts have exactly 0,1,2,3,4 constructs YES (sanity
    check that bat_score == sum of YES flags)
  - flags any unexpected values in EX/EMO/COG/MD (not YES/NO)

Edit INPUT_CSV below if the path differs.
"""

import pandas as pd

INPUT_CSV = "bat_posts_results.csv"
CONSTRUCT_COLS = ["EX", "EMO", "COG", "MD"]

df = pd.read_csv(INPUT_CSV)
n_total = len(df)
print(f"Total rows: {n_total:,}\n")

# ---- row_type breakdown ----
if "row_type" in df.columns:
    print("=" * 60)
    print("row_type breakdown")
    print("=" * 60)
    print(df["row_type"].value_counts(dropna=False).to_string())
    print()

# ---- unexpected values check ----
print("=" * 60)
print("Unique values per construct column (flagging anything not YES/NO)")
print("=" * 60)
for col in CONSTRUCT_COLS:
    if col not in df.columns:
        print(f"  {col}: COLUMN NOT FOUND")
        continue
    vals = df[col].value_counts(dropna=False)
    unexpected = [v for v in vals.index if str(v).strip().upper() not in ("YES", "NO")]
    print(f"  {col}: {dict(vals)}")
    if unexpected:
        print(f"    ^ WARNING unexpected values: {unexpected}")
print()

# ---- per-construct YES/NO counts and rates ----
print("=" * 60)
print("Per-construct counts and rates (of total rows)")
print("=" * 60)
summary_rows = []
for col in CONSTRUCT_COLS:
    if col not in df.columns:
        continue
    is_yes = df[col].astype(str).str.strip().str.upper() == "YES"
    n_yes = int(is_yes.sum())
    n_no = int((df[col].astype(str).str.strip().str.upper() == "NO").sum())
    n_other = n_total - n_yes - n_no
    pct_yes = round(n_yes / n_total * 100, 2)
    summary_rows.append({
        "construct": col,
        "YES_n": n_yes,
        "YES_pct": pct_yes,
        "NO_n": n_no,
        "other_n": n_other,
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

# ---- bat_score distribution ----
if "bat_score" in df.columns:
    print("=" * 60)
    print("bat_score distribution (0-4)")
    print("=" * 60)
    dist = df["bat_score"].value_counts().sort_index()
    print(dist.to_string())
    print(f"\nMean bat_score: {df['bat_score'].mean():.4f}")
    print(f"Median bat_score: {df['bat_score'].median():.1f}")
    print()

# ---- sanity check: does sum of YES flags == bat_score? ----
print("=" * 60)
print("Sanity check: sum(YES flags) vs bat_score")
print("=" * 60)
yes_sum = pd.Series(0, index=df.index)
for col in CONSTRUCT_COLS:
    if col in df.columns:
        yes_sum += (df[col].astype(str).str.strip().str.upper() == "YES").astype(int)

if "bat_score" in df.columns:
    mismatch = (yes_sum != df["bat_score"]).sum()
    print(f"Rows where sum(EX,EMO,COG,MD YES) != bat_score: {mismatch:,} "
          f"({mismatch / n_total * 100:.2f}%)")
    if mismatch > 0:
        print("  -> bat_score may not be a simple sum, or there's a parsing issue. "
              "Worth a quick look at a few mismatched rows.")
else:
    print("bat_score column not found, skipping sanity check.")
print()

# ---- co-occurrence: how many constructs YES per row ----
print("=" * 60)
print("Distribution of number of constructs YES per row (computed from flags)")
print("=" * 60)
print(yes_sum.value_counts().sort_index().to_string())

# Save summary to file
summary_df.to_csv("construct_counts_summary.csv", index=False)
print("\nSaved: construct_counts_summary.csv")

Total rows: 135,897

row_type breakdown
row_type
post    135897

Unique values per construct column (flagging anything not YES/NO)
  EX: {'NO': np.int64(126668), 'YES': np.int64(9050), 'ERROR': np.int64(179)}
    ^ WARNING unexpected values: ['ERROR']
  EMO: {'NO': np.int64(113362), 'YES': np.int64(22355), 'ERROR': np.int64(179), 'FUCK FUCK FUCK': np.int64(1)}
    ^ WARNING unexpected values: ['ERROR', 'FUCK FUCK FUCK']
  COG: {'NO': np.int64(131623), 'YES': np.int64(4093), 'ERROR': np.int64(179), 'The text does not describe difficulty with memory, focus, or decision-making at work, but rather a challenging situation that the author is trying to resolve.': np.int64(1), 'feeling overwhelmed by the amount of information available': np.int64(1)}
    ^ WARNING unexpected values: ['ERROR', 'The text does not describe difficulty with memory, focus, or decision-making at work, but rather a challenging situation that the author is trying to resolve.', 'feeling overwhelmed by the amount of info

# Adding timestamp column to the bat output 
Confirm the exact timestamp column name in master_posts_burnout_only.csv — likely created_utc, but possibly created, date, timestamp, etc.
Confirm post_id dtype/format matches in both files — if one is stored as a string like t3_8ln73a (raw Reddit fullname format) and the other is stripped to just 8ln73a, a merge will silently produce all-NaN dates instead of erroring, which is the exact "silent failure" pattern flagged in your project notes.

In [4]:
"""
Merge timestamp (and subreddit, if available) from master_posts_burnout_only.csv
into bat_posts_results.csv, joining on post_id.

Why the diagnostics matter: a merge on mismatched key formats (e.g. "8ln73a"
vs "t3_8ln73a") doesn't error -- it just silently produces NaN dates for
everything, which is exactly the kind of silent failure already flagged in
this project (the no_exit/default-confidence issue). So we check key overlap
BEFORE merging, not after.

EDIT the paths and column names below to match your files, then run.
"""

import pandas as pd

# ============================================================
# EDIT THESE PATHS
# ============================================================
MASTER_CSV = "master_posts_burnout_only.csv"
BAT_CSV = "bat_posts_results.csv"
OUTPUT_CSV = "bat_posts_results_with_dates.csv"

# Candidate timestamp column names to look for automatically in master file
TIMESTAMP_CANDIDATES = ["created_utc", "created", "date", "timestamp",
                         "created_date", "post_date", "datetime"]
# Candidate subreddit column names
SUBREDDIT_CANDIDATES = ["subreddit", "sub", "subreddit_name"]
# Candidate id column names in the master file (the bat file always uses post_id)
MASTER_ID_CANDIDATES = ["post_id", "id", "post_uid"]
# Extra columns to bring over if present (precomputed convenience fields)
EXTRA_COLS_TO_MERGE = ["created_date", "year", "month", "day_of_week", "hour", "year_month"]


def find_column(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            print(f"  Found {label} column: '{c}'")
            return c
    print(f"  WARNING: no {label} column found among {candidates}")
    print(f"  Actual columns in master file: {list(df.columns)}")
    return None


def diagnose_key_overlap(master_df, bat_df, key="post_id"):
    print("=" * 60)
    print(f"PRE-MERGE DIAGNOSTIC: checking '{key}' overlap")
    print("=" * 60)

    print(f"master_df['{key}'] dtype: {master_df[key].dtype}")
    print(f"bat_df['{key}'] dtype:    {bat_df[key].dtype}")

    m_sample = master_df[key].dropna().astype(str).head(5).tolist()
    b_sample = bat_df[key].dropna().astype(str).head(5).tolist()
    print(f"master_df['{key}'] sample values: {m_sample}")
    print(f"bat_df['{key}'] sample values:    {b_sample}")

    master_keys = set(master_df[key].astype(str).str.strip())
    bat_keys = set(bat_df[key].astype(str).str.strip())

    overlap = master_keys & bat_keys
    only_master = master_keys - bat_keys
    only_bat = bat_keys - master_keys

    print(f"\nUnique {key} in master:  {len(master_keys):,}")
    print(f"Unique {key} in bat:     {len(bat_keys):,}")
    print(f"Overlap (exact string match): {len(overlap):,}")
    print(f"In master only: {len(only_master):,}")
    print(f"In bat only:    {len(only_bat):,}")

    overlap_pct = len(overlap) / max(len(bat_keys), 1) * 100
    print(f"\n{overlap_pct:.1f}% of bat_df post_ids found a match in master_df.")

    if overlap_pct < 50:
        print("\n*** LOW OVERLAP WARNING ***")
        print("This usually means a format mismatch, e.g. one file stores")
        print("post_id with a Reddit fullname prefix ('t3_xxxxx') and the")
        print("other stores it stripped ('xxxxx'). Sample comparison:")
        for v in b_sample[:3]:
            print(f"  bat post_id    : {v}")
        for v in m_sample[:3]:
            print(f"  master post_id : {v}")
        print("\nIf you see a 't3_' prefix on one side and not the other,")
        print("strip it before merging (see strip_prefix flag below).")
    else:
        print("\nOverlap looks healthy. Proceeding with merge.")

    return overlap_pct


def main(strip_t3_prefix="auto"):
    print(f"Loading {MASTER_CSV} ...")
    master_df = pd.read_csv(MASTER_CSV)
    print(f"  {len(master_df):,} rows, columns: {list(master_df.columns)}\n")

    print(f"Loading {BAT_CSV} ...")
    bat_df = pd.read_csv(BAT_CSV)
    print(f"  {len(bat_df):,} rows\n")

    # Master file uses 'id' rather than 'post_id' -- detect and rename
    master_id_col = None
    for c in MASTER_ID_CANDIDATES:
        if c in master_df.columns:
            master_id_col = c
            break
    if master_id_col is None:
        raise ValueError(f"No id-like column found in master file among {MASTER_ID_CANDIDATES}. "
                          f"master cols: {list(master_df.columns)}")
    if master_id_col != "post_id":
        print(f"Master file uses '{master_id_col}' as its id column; renaming to 'post_id' for merge.")
        master_df = master_df.rename(columns={master_id_col: "post_id"})

    if "post_id" not in bat_df.columns:
        raise ValueError(f"post_id column missing from bat file. bat cols: {list(bat_df.columns)}")

    ts_col = find_column(master_df, TIMESTAMP_CANDIDATES, "timestamp")
    sub_col = find_column(master_df, SUBREDDIT_CANDIDATES, "subreddit")
    print()

    # Normalize post_id to plain string for comparison
    master_df["post_id"] = master_df["post_id"].astype(str).str.strip()
    bat_df["post_id"] = bat_df["post_id"].astype(str).str.strip()

    # Optionally auto-detect/strip t3_ prefix mismatch
    m_has_prefix = master_df["post_id"].str.startswith("t3_").mean() > 0.5
    b_has_prefix = bat_df["post_id"].str.startswith("t3_").mean() > 0.5
    if strip_t3_prefix == "auto":
        if m_has_prefix and not b_has_prefix:
            print("Detected: master has 't3_' prefix, bat does not. Stripping from master.")
            master_df["post_id"] = master_df["post_id"].str.replace("^t3_", "", regex=True)
        elif b_has_prefix and not m_has_prefix:
            print("Detected: bat has 't3_' prefix, master does not. Stripping from bat.")
            bat_df["post_id"] = bat_df["post_id"].str.replace("^t3_", "", regex=True)

    overlap_pct = diagnose_key_overlap(master_df, bat_df, key="post_id")

    if overlap_pct < 10:
        print("\nOverlap too low to proceed safely. Stopping before merge.")
        print("Inspect the sample post_id values above and adjust manually.")
        return

    # Build the slim master subset to merge in
    cols_to_merge = ["post_id"]
    rename_map = {}
    if ts_col:
        cols_to_merge.append(ts_col)
        rename_map[ts_col] = "created_utc"
    if sub_col:
        cols_to_merge.append(sub_col)
        rename_map[sub_col] = "subreddit"
    for extra_col in EXTRA_COLS_TO_MERGE:
        if extra_col in master_df.columns and extra_col not in cols_to_merge:
            cols_to_merge.append(extra_col)

    master_slim = master_df[cols_to_merge].drop_duplicates(subset="post_id")
    master_slim = master_slim.rename(columns=rename_map)

    merged = bat_df.merge(master_slim, on="post_id", how="left")

    n_matched = merged["created_utc"].notna().sum() if "created_utc" in merged.columns else 0
    n_total = len(merged)
    print("\n" + "=" * 60)
    print("MERGE RESULT")
    print("=" * 60)
    print(f"Total rows in bat_df: {n_total:,}")
    if "created_utc" in merged.columns:
        print(f"Rows with a matched created_utc: {n_matched:,} ({n_matched/n_total*100:.1f}%)")
        print(f"Rows with NO match (created_utc is null): {n_total - n_matched:,}")

    merged.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved merged file -> {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Loading master_posts_burnout_only.csv ...
  135,897 rows, columns: ['id', 'subreddit', 'author', 'created_date', 'created_utc', 'year', 'month', 'day_of_week', 'hour', 'year_month', 'title', 'selftext', 'text', 'cleaned_text', 'original_text_len', 'cleaned_text_len', 'word_count', 'score', 'upvote_ratio', 'num_comments', 'link_flair_text', 'permalink', 'url', 'text_processed', 'predicted_label', 'prediction_confidence']

Loading bat_posts_results.csv ...
  135,897 rows

Master file uses 'id' as its id column; renaming to 'post_id' for merge.
  Found timestamp column: 'created_utc'
  Found subreddit column: 'subreddit'

PRE-MERGE DIAGNOSTIC: checking 'post_id' overlap
master_df['post_id'] dtype: object
bat_df['post_id'] dtype:    object
master_df['post_id'] sample values: ['8ln73a', 'b3rtqm', 'b88bq5', 'b9xiaz', 'bpo012']
bat_df['post_id'] sample values:    ['8ln73a', 'b3rtqm', 'b88bq5', 'b9xiaz', 'bpo012']

Unique post_id in master:  135,897
Unique post_id in bat:     135,897
Overlap (

In [5]:
#let's do a check to the new file:  bat_posts_results_with_dates.csv
import pandas as pd
df = pd.read_csv("bat_posts_results_with_dates.csv")
print(df.columns.tolist())
print(df.head(3).to_string())
print(df.dtypes)

['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'created_utc', 'subreddit', 'created_date', 'year', 'month', 'day_of_week', 'hour', 'year_month']
  row_type post_id  comment_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  text  triage  na_subtype  triage_reason  EX EMO COG  MD  bat_score    

In [2]:
'''

So there’s an issue, I need to filter the 20 samples that human labeled, from the updated BAT prompt run
Result saved in Redditrun_june folder - bat_posts_results.csv [filter the ones from the given to Ji Hyun]



'''


import pandas as pd

path1 = "labeling_BAT_human_hyun.csv"   # <-- human-labeled 20 samples
path2 = "bat_posts_results.csv"         # <-- full updated LLM BAT run
output_path = "bat_post20_LLM.csv"

human_df = pd.read_csv(path1, dtype=str)
llm_df   = pd.read_csv(path2, dtype=str)

print(f"Human-labeled file: {len(human_df)} rows, columns: {list(human_df.columns)}")
print(f"LLM results file:   {len(llm_df)} rows, columns: {list(llm_df.columns)}")

# Identify the post_id column in each (case-insensitive match, in case naming differs)
def find_id_col(df, name="post_id"):
    for col in df.columns:
        if col.strip().lower() == name:
            return col
    raise ValueError(f"Could not find a '{name}' column. Available: {list(df.columns)}")

human_id_col = find_id_col(human_df)
llm_id_col   = find_id_col(llm_df)

human_ids = set(human_df[human_id_col].astype(str).str.strip())
print(f"\nUnique human-labeled post_ids: {len(human_ids)}")

filtered = llm_df[llm_df[llm_id_col].astype(str).str.strip().isin(human_ids)]

print(f"Matched rows in LLM results: {len(filtered)}")

# Flag any human-labeled ids that did NOT find a match (helps catch typos/mismatches)
matched_ids = set(filtered[llm_id_col].astype(str).str.strip())
missing = human_ids - matched_ids
if missing:
    print(f"\n⚠ {len(missing)} human-labeled post_id(s) not found in LLM results:")
    for m in missing:
        print(f"   {m}")

n_missing = len(missing)

if n_missing > 0:
    # Pool to sample extra rows from: LLM rows NOT already matched
    remaining_pool = llm_df[~llm_df[llm_id_col].astype(str).str.strip().isin(human_ids)]
    n_to_sample = min(n_missing, len(remaining_pool))
    extra_rows = remaining_pool.sample(n=n_to_sample, random_state=None)

    print(f"\nRandomly sampling {n_to_sample} extra row(s) to replace the missing match(es)...")
    filtered = pd.concat([filtered, extra_rows], ignore_index=True)

print(f"\nFinal output row count: {len(filtered)} (target: {len(human_ids)})")

filtered.to_csv(output_path, index=False)
print(f"Saved {len(filtered)} rows to: {output_path}")

Human-labeled file: 20 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'EX', 'EMO', 'COG', 'MD', 'Reason', 'Context (Y/N)', 'Context Reason/ (things were not explicitly mentioned in the prompt??)']
LLM results file:   135897 rows, columns: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning']

Unique human-labeled post_ids: 20
Matched rows in LLM results: 17

⚠ 3 human-labeled post_id(s) not found in LLM results:
   1gjdtol
   1988cgl
   1cc4iax

Randomly sampling 3 extra row(s) to replace the missing match(es)...

Final output row count: 20 (target: 20)
Saved 20 rows to: bat_post20_LLM.csv
